# 01: DOWNLOAD GOOGLE STREET VIEW IMAGES

This notebook downloads Google Street View images along pedestrian network edges for selected wards in Ho Chi Minh City. It also saves the network graph for routing.

## MODULE SETUP

In [ ]:
# Mount Google Drive.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

# Set working directory to project folder.
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/hot_hem"
os.chdir(BASE_DIR)

print(f"Working directory: {BASE_DIR}")

In [ ]:
# Install required packages.
!pip install contextily osmnx googlemaps -q

## IMPORT SETUP

In [ ]:
from pathlib import Path
import json
import time
import requests

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx
import osmnx as ox
import networkx as nx
from shapely.geometry import LineString, Point
from pyproj import Transformer
from tqdm import tqdm

## PATH CONFIGURATION

In [ ]:
# Input paths.
BOUNDARY_FILE = Path("data/inputs/boundaries/aoi_wards.geojson")

# Output paths for network data.
NETWORK_DIR = Path("data/processing/network")
NETWORK_GRAPHML = NETWORK_DIR / "hcmc_pedestrian_network.graphml"
NODES_CSV = NETWORK_DIR / "network_nodes.csv"
EDGES_CSV = NETWORK_DIR / "network_edges.csv"

# Output paths for GSV data.
GSV_DIR = Path("data/processing/gsv")
META_CSV = GSV_DIR / "metadata.csv"
CHECKPOINT_FILE = GSV_DIR / "checkpoint.json"
GSV_GEOJSON = GSV_DIR / "gsv_sample_points.geojson"
THUMBNAIL_HTML = GSV_DIR / "gsv_thumbnails.html"

# Output path for images.
IMAGE_DIR = Path("data/processing/images")

# Create directories.
NETWORK_DIR.mkdir(parents = True, exist_ok = True)
GSV_DIR.mkdir(parents = True, exist_ok = True)
IMAGE_DIR.mkdir(parents = True, exist_ok = True)

print("PATH CONFIGURATION")
print(f"Boundary file: {BOUNDARY_FILE}")
print(f"Network directory: {NETWORK_DIR}")
print(f"GSV directory: {GSV_DIR}")
print(f"Image directory: {IMAGE_DIR}")

## PARAMETERS

In [ ]:
# Google Street View API key.
API_KEY = "API_KEY"

# Sampling interval in meters.
DIST_INTERVAL = 50

# Buffer distance in meters for network extraction.
BUFFER_DISTANCE = 500

# Target wards for GSV image collection.
# District 1: Ben Thanh, Co Giang.
# District 2: Thao Dien, An Khanh.
# District 8: Ward 5, Ward 6.
SELECTED_WARDS = [
    "Ben Thanh",
    "Co Giang",
    "Thao Dien",
    "An Khanh",
    "Ward 5",
    "Ward 6"
]

# Coordinate reference systems.
CRS_WGS84 = "EPSG:4326"
CRS_VIETNAM = "EPSG:3405"

print("PARAMETERS")
print(f"Sampling interval: {DIST_INTERVAL}m")
print(f"Buffer distance: {BUFFER_DISTANCE}m")
print(f"Target wards: {SELECTED_WARDS}")

## LOAD WARD BOUNDARIES

In [ ]:
# Load ward boundary polygons.
wards = gpd.read_file(BOUNDARY_FILE)

print(f"Loaded {len(wards)} ward boundaries.")
print(f"CRS: {wards.crs}")
print(f"Columns: {list(wards.columns)}")

In [ ]:
# Filter to target wards for GSV collection.
gsv_wards = wards[wards["ENGNAME_3"].isin(SELECTED_WARDS)].copy()
gsv_wards = gsv_wards.reset_index(drop = True)

print(f"Filtered to {len(gsv_wards)} target wards for GSV collection.")
print(gsv_wards[["ENGNAME_2", "ENGNAME_3"]])

## CREATE BUFFERED BOUNDARIES

In [ ]:
# Buffer all wards by 500m for full network extraction.
# Project to Vietnam CRS for accurate distance calculation.
wards_buff = wards.to_crs(CRS_VIETNAM)
wards_buff["geometry"] = wards_buff.geometry.buffer(BUFFER_DISTANCE)
wards_buff = wards_buff.to_crs(CRS_WGS84)

print(f"{BUFFER_DISTANCE}m buffer for all wards.")

In [ ]:
# Buffer GSV target wards by 500m.
gsv_wards_buff = gsv_wards.to_crs(CRS_VIETNAM).copy()
gsv_wards_buff["geometry"] = gsv_wards_buff.geometry.buffer(BUFFER_DISTANCE)
gsv_wards_buff = gsv_wards_buff.to_crs(CRS_WGS84)

print(f"{BUFFER_DISTANCE}m buffer for GSV target wards.")

## VISUALIZE STUDY AREA

In [ ]:
# Plot all wards with target wards highlighted.
fig, ax = plt.subplots(1, 1, figsize = (12, 10))
fig.patch.set_facecolor("white")

# Plot all ward buffers.
wards_buff.plot(ax = ax, facecolor = "lightgray", edgecolor = "gray", alpha = 0.5)

# Highlight GSV target wards.
gsv_wards_buff.plot(ax = ax, facecolor = "lightblue", edgecolor = "blue", alpha = 0.7)

# Add basemap.
ctx.add_basemap(ax, crs = wards_buff.crs.to_string(), source = ctx.providers.CartoDB.Positron)

ax.set_title("Study Area: Districts 1, 2, 8 with Target Wards Highlighted")
ax.axis("off")
plt.tight_layout()
plt.show()

## EXTRACT PEDESTRIAN NETWORK

In [ ]:
# Get pedestrian network for all districts with buffer.
# This is used for routing across full study area.
print("Extracting pedestrian network for full study area.")

G_wards = ox.graph_from_polygon(
    wards_buff.union_all(),
    network_type = "walk"
)

# Convert to GeoDataFrames.
ward_nodes, ward_edges = ox.graph_to_gdfs(G_wards, nodes = True, edges = True)

print(f"Full network: {len(G_wards.nodes)} nodes, {len(G_wards.edges)} edges.")

In [ ]:
# Get pedestrian network for GSV target wards.
# Process each ward separately then combine.
print("Extracting pedestrian network for GSV target wards.")

graphs = []

for idx, ward in gsv_wards.iterrows():
    ward_name = ward["ENGNAME_3"]
    
    try:
        print(f"Processing {ward_name}.")
        
        # Use buffered polygon.
        geom = gsv_wards_buff.geometry.iloc[idx]
        
        G = ox.graph_from_polygon(geom, network_type = "walk")
        graphs.append(G)
        
        print(f"Extracted {len(G.nodes)} nodes from {ward_name}.")
        
    except Exception as e:
        print(f"Failed to extract {ward_name}: {type(e).__name__}: {e}")

print(f"Total graphs extracted: {len(graphs)}.")

if len(graphs) == 0:
    raise RuntimeError("No graphs extracted. Check ward boundaries.")

In [ ]:
# Combine ward graphs into single network.
G_gsv_wards = nx.compose_all(graphs)

# Add forward-direction bearings to edges.
G_gsv_wards = ox.bearing.add_edge_bearings(G_gsv_wards)

# Convert to GeoDataFrames.
gsv_ward_nodes, gsv_ward_edges = ox.graph_to_gdfs(G_gsv_wards, nodes = True, edges = True)

print(f"Combined GSV network: {len(gsv_ward_nodes)} nodes, {len(gsv_ward_edges)} edges.")

## SAVE NETWORK DATA

In [ ]:
# Save full network graph for routing.
ox.save_graphml(G_wards, NETWORK_GRAPHML)
print(f"Network graph saved to: {NETWORK_GRAPHML}")

# Save nodes and edges as CSV for inspection.
ward_nodes.to_csv(NODES_CSV)
print(f"Network nodes saved to: {NODES_CSV}")

ward_edges.to_csv(EDGES_CSV)
print(f"Network edges saved to: {EDGES_CSV}")

print(f"Network has {len(ward_nodes)} nodes and {len(ward_edges)} edges.")

## SAMPLE GSV POINTS ALONG EDGES

In [ ]:
# Reproject edges to meter-based CRS for accurate distance sampling.
gsv_ward_edges_proj = gsv_ward_edges.to_crs(CRS_VIETNAM)

# Verify projected CRS.
assert gsv_ward_edges_proj.crs.is_projected, "Expected projected CRS for distance calculations."

print(f"Reprojected {len(gsv_ward_edges_proj)} edges to {CRS_VIETNAM}.")

In [ ]:
# Sample points along edges at specified interval.
# Each point gets heading aligned with edge bearing.
transformer = Transformer.from_crs(gsv_ward_edges_proj.crs, CRS_WGS84, always_xy = True)

gsv_points = []
uid_counter = 0

for idx, row in tqdm(gsv_ward_edges_proj.iterrows(), total = len(gsv_ward_edges_proj), desc = "Sampling points"):
    line = row.geometry
    length = int(line.length)
    
    # Skip short edges.
    if length < DIST_INTERVAL:
        continue
    
    # Get forward bearing from OSMnx edge attribute.
    forward_bearing = float(row["bearing"])
    
    # Sample points at interval.
    for dist in range(0, length, DIST_INTERVAL):
        pt = line.interpolate(dist)
        lon, lat = transformer.transform(pt.x, pt.y)
        
        gsv_points.append({
            "uid": uid_counter,
            "lat": lat,
            "lon": lon,
            "heading": forward_bearing
        })
        uid_counter += 1

print(f"Generated {len(gsv_points)} GSV sample points.")

In [ ]:
# Convert to GeoDataFrame and assign ward attributes.
gdf_gsv = gpd.GeoDataFrame(
    gsv_points,
    geometry = [Point(p["lon"], p["lat"]) for p in gsv_points],
    crs = CRS_WGS84
)

# Ensure CRS matches for spatial join.
if gdf_gsv.crs != gsv_wards_buff.crs:
    gdf_gsv = gdf_gsv.to_crs(gsv_wards_buff.crs)

# Spatial join with buffered wards to assign administrative info.
gdf_gsv = gpd.sjoin(
    gdf_gsv,
    gsv_wards_buff[["ENGNAME", "ENGNAME_1", "ENGNAME_2", "ENGNAME_3", "geometry"]],
    how = "left",
    predicate = "intersects"
).drop(columns = "index_right", errors = "ignore")

# Ensure UID is integer.
gdf_gsv["uid"] = gdf_gsv["uid"].astype(int)

# Save sample points as GeoJSON.
gdf_gsv.to_file(GSV_GEOJSON, driver = "GeoJSON")
print(f"GSV sample points saved to: {GSV_GEOJSON}")
print(f"Total points: {len(gdf_gsv)}")

## GSV DOWNLOAD HELPER FUNCTION

In [ ]:
def download_gsv(lat, lon, heading, save_path):
    """
    Download Google Street View image at specified location.
    Heading is aligned with road forward bearing from OSMnx.
    Returns True if successful, False otherwise.
    """
    url = (
        "https://maps.googleapis.com/maps/api/streetview"
        f"?size=2048x1024"
        f"&location={lat},{lon}"
        f"&heading={heading}"
        f"&pitch=0"
        f"&fov=120"
        f"&key={API_KEY}"
    )
    
    try:
        r = requests.get(url, timeout = 10)
        
        if r.status_code == 200:
            with open(save_path, "wb") as f:
                f.write(r.content)
            return True
        else:
            print(f"Failed to download {lat},{lon}: status {r.status_code}")
            return False
            
    except Exception as e:
        print(f"Error downloading {lat},{lon}: {e}")
        return False

## LOAD CHECKPOINT AND EXISTING METADATA

In [ ]:
# Load checkpoint if exists.
if CHECKPOINT_FILE.exists():
    with open(CHECKPOINT_FILE, "r") as f:
        checkpoint = json.load(f)
        last_done = checkpoint.get("last_downloaded_uid", -1)
else:
    last_done = -1

print(f"Last successfully downloaded UID: {last_done}")

# Load existing metadata if exists.
if META_CSV.exists():
    df_existing = pd.read_csv(META_CSV)
    existing_uids = set(df_existing["uid"].astype(int).tolist())
    metadata_records = df_existing.to_dict("records")
    print(f"Loaded {len(df_existing)} existing metadata records.")
else:
    existing_uids = set()
    metadata_records = []
    print("No existing metadata found. Starting fresh.")

## DOWNLOAD GSV IMAGES

In [ ]:
# Main download loop with checkpoint recovery.
for idx, row in tqdm(gdf_gsv.iterrows(), total = len(gdf_gsv), desc = "Downloading GSV"):
    uid = int(row["uid"])
    
    # Skip if already processed.
    if uid <= last_done or uid in existing_uids:
        continue
    
    lat = row["lat"]
    lon = row["lon"]
    heading = row["heading"]
    
    # Normalize district and ward labels.
    country_label = str(row["ENGNAME"]) if pd.notnull(row["ENGNAME"]) else "Vietnam"
    city_label = str(row["ENGNAME_1"]) if pd.notnull(row["ENGNAME_1"]) else "Ho Chi Minh"
    
    district_label = (
        str(row["ENGNAME_2"]).strip().lower().replace("district ", "")
        if pd.notnull(row["ENGNAME_2"]) else "unknown"
    )
    
    ward_label = (
        str(row["ENGNAME_3"]).strip().lower().replace(" ", "_")
        if pd.notnull(row["ENGNAME_3"]) else "unknown"
    )
    
    # Create output directory structure.
    # Format: data/processing/images/district_X/ward_Y/original/
    out_dir = IMAGE_DIR / f"district_{district_label}" / ward_label / "original"
    out_dir.mkdir(parents = True, exist_ok = True)
    
    # Create image filename.
    img_name = f"gsv_{uid}.jpg"
    save_path = out_dir / img_name
    
    # Skip if image already exists.
    if save_path.exists():
        continue
    
    # Download image.
    success = download_gsv(lat, lon, heading, str(save_path))
    
    if not success:
        continue
    
    # Append metadata record.
    metadata_records.append({
        "uid": uid,
        "lat": lat,
        "lon": lon,
        "heading": heading,
        "country": country_label,
        "city": city_label,
        "district": row["ENGNAME_2"],
        "ward": row["ENGNAME_3"],
        "image_path": str(save_path)
    })
    existing_uids.add(uid)
    
    # Update checkpoint.
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump({"last_downloaded_uid": uid}, f)
    
    # Periodically flush metadata to disk.
    if uid % 100 == 0:
        df_meta_flush = pd.DataFrame(metadata_records)
        df_meta_flush.to_csv(META_CSV, index = False)
        print(f"Checkpoint saved at UID {uid}.")
    
    # Rate limiting.
    time.sleep(0.1)

# Final metadata save.
df_meta = pd.DataFrame(metadata_records)
df_meta.to_csv(META_CSV, index = False)

print(f"Download complete. Saved {len(df_meta)} records to {META_CSV}.")

## GENERATE THUMBNAILS AND ALIGN GEOJSON

In [ ]:
# Add thumbnail HTML column.
df_meta["thumbnail"] = df_meta["image_path"].apply(
    lambda x: f'<img src="{x}" width="100">' if Path(x).exists() else "Missing"
)

# Save thumbnail HTML for visual inspection.
df_meta.to_html(THUMBNAIL_HTML, escape = False)
print(f"Thumbnail HTML saved to: {THUMBNAIL_HTML}")

# Align geometries by UID.
gdf_gsv_uid = gdf_gsv.set_index("uid")
df_meta_uid = df_meta.set_index("uid")

# Join metadata with geometry.
df_meta_uid = df_meta_uid.join(gdf_gsv_uid[["geometry"]], how = "inner")
gdf_meta = gpd.GeoDataFrame(df_meta_uid, geometry = "geometry", crs = gdf_gsv.crs)

# Save aligned GeoJSON.
gdf_meta.to_file(GSV_GEOJSON, driver = "GeoJSON")
print(f"Aligned GeoJSON saved to: {GSV_GEOJSON}")
print(f"Total points with images: {len(gdf_meta)}")

## SUMMARY

In [ ]:
print("DOWNLOAD SUMMARY")
print(f"Total GSV images downloaded: {len(df_meta)}")
print(f"Network nodes: {len(ward_nodes)}")
print(f"Network edges: {len(ward_edges)}")
print("")
print("Output files:")
print(f"Network graph: {NETWORK_GRAPHML}")
print(f"Network nodes: {NODES_CSV}")
print(f"Network edges: {EDGES_CSV}")
print(f"GSV metadata: {META_CSV}")
print(f"GSV GeoJSON: {GSV_GEOJSON}")
print(f"Thumbnails: {THUMBNAIL_HTML}")
print(f"Images: {IMAGE_DIR}")